## Model routing lab

Playground to try routing to an AI Foundry backend based on the requested model (Chat Completions and Responses API).

This version adds a fourth backend to contrast the current Foundry resource model with the legacy hub-based topology. The legacy project is connected to a classic Azure OpenAI account and exposes the `legacy-gpt-4o` deployment through the same APIM endpoint.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/) — run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`... 


### 🗺️ Backend topology & scenarios

The full architecture diagram, backend/deployment table, routing rules, and the two load-balancing setups (deployment-level failover and backend-pool failover) are documented in **[README.md](README.md)**.

In short: a single APIM endpoint routes by requested model to region-specific Foundry backends, and demonstrates two PTU→TPM spillover patterns — `gpt-5-mini` (deployment-level failover on one Foundry resource) and `gpt-5.4-mini` (priority backend pool + circuit breaker across two Foundry resources).


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the models and versions according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "eastus"
utils.print_ok(resource_group_name)

aiservices_config = [{"name": "foundry1", "location": "swedencentral"},
                    {"name": "foundry2", "location": "centralus"},
                    {"name": "foundry3", "location": "eastus2"},
                    # Scenario 2 Foundry resources: one gpt-5.4-mini deployment each, load balanced by a priority pool
                    {"name": "foundry6", "location": "eastus2", "priority": 1, "circuitBreaker": True},
                    {"name": "foundry7", "location": "eastus2", "priority": 2, "circuitBreaker": True}]

models_config = [{"name": "gpt-4.1", "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 20, "aiservice": "foundry1"},
                 # Scenario 1: two gpt-5-mini deployments on the SAME Foundry resource (PTU-mimic primary + TPM spillover).
                 # "model" is the underlying model; "name" is the (distinct) deployment name. Primary capacity is low so it 429s and spills over.
                 {"name": "gpt-5-mini-ptu", "model": "gpt-5-mini", "publisher": "OpenAI", "version": "2025-08-07", "sku": "GlobalStandard", "capacity": 1, "aiservice": "foundry2"},
                 {"name": "gpt-5-mini-tpm", "model": "gpt-5-mini", "publisher": "OpenAI", "version": "2025-08-07", "sku": "GlobalStandard", "capacity": 20, "aiservice": "foundry2"},
                 {"name": "gpt-5-nano", "publisher": "OpenAI", "version": "2025-08-07", "sku": "GlobalStandard", "capacity": 20, "aiservice": "foundry2"},
                 {"name": "gpt-5-pro", "publisher": "OpenAI", "version": "2025-10-06", "sku": "GlobalStandard", "capacity": 20, "aiservice": "foundry3"},
                 {"name": "gpt-5.6-terra", "publisher": "OpenAI", "version": "2026-07-09", "sku": "GlobalStandard", "capacity": 20, "aiservice": "foundry3"},
                 # Scenario 2: same deployment name on two separate Foundry resources, load balanced by the priority pool below.
                 {"name": "gpt-5.4-mini", "publisher": "OpenAI", "version": "2026-03-17", "sku": "GlobalStandard", "capacity": 1, "aiservice": "foundry6"},
                 {"name": "gpt-5.4-mini", "publisher": "OpenAI", "version": "2026-03-17", "sku": "GlobalStandard", "capacity": 20, "aiservice": "foundry7"}]

# Scenario 2 priority pool: foundry6 (primary) -> foundry7 (spillover) when foundry6's circuit breaker trips on 429
backend_pools_config = [{"name": "gpt54mini-pool",
                         "description": "Priority PTU/TPM spillover pool for gpt-5.4-mini",
                         "services": [{"backend": "foundry6", "priority": 1},
                                      {"backend": "foundry7", "priority": 2}]}]

legacy_foundry_config = {"backendName": "foundry4",
                         "location": "eastus2",
                         "deploymentName": "legacy-gpt-4o",
                         "modelName": "gpt-4o",
                         "modelVersion": "2024-11-20",
                         "modelSku": "GlobalStandard",
                         "capacity": 20}

# foundry5 is a new AIServices Foundry that does NOT deploy its own model -- it exposes
# legacy-gpt-4o through an AzureOpenAI connection to foundry4 (control-plane discovery only).
foundry5_config = {"backendName": "foundry5",
                   "location": "eastus2"}

apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

inference_api_path = "inference"  # path to the inference API in the APIM service
inference_api_type = "PassThrough"  # options: AzureOpenAI, AzureAI, OpenAI, PassThrough
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations. 

In [ ]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "legacyFoundryConfig": { "value": legacy_foundry_config },
        "foundry5Config": { "value": foundry5_config },
        "backendPoolsConfig": { "value": backend_pools_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the required outputs from the Bicep deployment.

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key") # default api key to the first subscription key



<a id='sdk'></a>
### 🧪 Test the API using the Azure OpenAI Python SDK
#### Chat Completions


In [ ]:
import time
from openai import AzureOpenAI
messages=[
        {'role': 'user', 'content': 'which model are you using?'},
]
start_time = time.time()
client = AzureOpenAI(
    azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
    api_key=api_key,
    api_version=inference_api_version
)
try:
    # gpt-5-mini -> foundry2 deployment failover (ptu->tpm); gpt-5.4-mini -> foundry6/foundry7 priority pool.
    # gpt-5-pro is Responses-API-only, so it is exercised in the Responses API cell below.
    for model in ['gpt-4.1', 'gpt-5-mini', 'gpt-5-nano', 'gpt-5.4-mini', 'gpt-5.6-terra', 'legacy-gpt-4o']:
        completion = client.chat.completions.with_raw_response.create(model=model, messages=messages)
        # print("headers ", completion.headers)
        print("x-ms-region: ", completion.headers.get("x-ms-region")) # this header is useful to determine the region of the backend that served the request

        completion = completion.parse()

        print(f"Model: {completion.model} 💬: {completion.choices[0].message.content}\n")
except Exception as e:
    print(f"Error: {e}")

#### Responses API
`gpt-4.1`, `gpt-5-mini`, `gpt-5-nano`, `gpt-5-pro`, and `gpt-5.6-terra` all support the Responses API. `gpt-5-pro` is a **Responses-API-only** reasoning model, so it appears here but not in the Chat Completions test. The legacy `gpt-4o` compatibility route is exercised through Chat Completions above.

In [ ]:
start_time = time.time()
input_message = "which model are you using?"

client = AzureOpenAI(
    azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
    api_key=api_key,
    api_version=inference_api_version
)
try:
    for model in ['gpt-4.1', 'gpt-5-mini', 'gpt-5-nano', 'gpt-5-pro', 'gpt-5.4-mini', 'gpt-5.6-terra']:
        responses = client.responses.with_raw_response.create(model=model, input=input_message)
        # print("headers ", responses.headers)
        print("x-ms-region: ", responses.headers.get("x-ms-region"))
        output = responses.parse()
        print(f"Model: {output.model} 💬: {output.output_text}\n")

except Exception as e:
    print(f"Error: {e}")

### 🔀 Direct vs. APIM — all six call shapes for `legacy-gpt-4o`

The classic Azure OpenAI account (`foundry4`) is reachable two ways, over two different API surfaces. These are **independent choices** — fronting the account with APIM does *not* require switching from Chat Completions to Responses.

| # | Scenario | Host | Surface | Credential | Expected |
|---|---|---|---|---|---|
| 1 | Direct, unauthenticated | classic AOAI | Chat Completions | none | **401** — no identity established |
| 2 | Direct, **account key** | classic AOAI | Chat Completions | `api-key:` AOAI account key | **403** (docs say 401) — key auth disabled |
| 3 | Direct, Entra token | classic AOAI | Chat Completions | `Authorization: Bearer` | **200** if you hold `Cognitive Services OpenAI User`, else **401/403** |
| 4 | Direct, Entra token | classic AOAI | Responses | `Authorization: Bearer` | same as 3 |
| 5 | Via APIM | APIM gateway | Chat Completions | `api-key:` APIM subscription key | **200** |
| 6 | Via APIM | APIM gateway | Responses | `api-key:` APIM subscription key | **200** |

Things the cell below demonstrates:

- **`api-key` means two completely different things.** In scenario 2 it is an *Azure OpenAI account key* sent to the AOAI data plane. In scenarios 5 and 6 it is an *APIM subscription key* sent to the gateway. Same header name, unrelated credentials, different issuers, different validators.
- **`disableLocalAuth: true` is enforced two ways.** The control-plane `listKeys` call is blocked (so scenario 2 can't even retrieve a real key), and the data plane rejects key auth outright. Scenario 2 prints both halves of that.
- **The deployment name travels differently per surface.** Chat Completions puts it in the **URL path** (`/openai/deployments/legacy-gpt-4o/chat/completions`); Responses puts it **only in the body `model` field**. There is no `/openai/deployments/{d}/responses` route — it does not exist.
- **`disableLocalAuth: true` does not force traffic through APIM.** It disables *key* auth only — scenarios 3 and 4 bypass APIM entirely using Entra, which is exactly why RBAC scoping matters.

> **Why scenario 2 returns 403 and not 401.** Microsoft's [disable-local-auth](https://learn.microsoft.com/azure/ai-services/disable-local-auth) guidance says to expect **401** `Access denied due to invalid subscription key or wrong API endpoint` — but that is also the documented response for an *invalid key value*. Since the key here is a placeholder, a 401 would mean the gateway validated it and found it wrong. A **403** means the gateway never compared the value at all: it recognized the credential *type* and refused the *method*. That is the semantically correct answer (identity/method recognized but forbidden), and it matches how Azure Storage documents the same "Shared Key disallowed" case. Treat the 403 as observed-but-undocumented. Every other 403 cause — network ACLs, Azure Policy, content filtering, missing custom subdomain — is credential-independent and would have hit scenario 1 too, which returns 401.

> **If scenarios 3 and 4 return 401/403, that is expected**, not a bug — [main.bicep](main.bicep) grants `Cognitive Services OpenAI User` on `foundry4` to the *APIM* identity and to `foundry5`, not to you. To try the direct path yourself:
> ```
> az role assignment create --assignee <your-object-id> --role "Cognitive Services OpenAI User" --scope <legacyOpenAIAccountId>
> ```
> Note that `Cognitive Services Contributor` will **not** work — it cannot make inference calls with Entra ID.


In [ ]:
import requests

# Resolve the classic Azure OpenAI account (foundry4) so we can call it directly, bypassing APIM.
dep = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}",
                "Retrieved deployment outputs", "Failed to retrieve deployment outputs")
legacy_account_id = utils.get_deployment_output(dep, 'legacyOpenAIAccountId', 'Legacy AOAI account id')

# 'az cognitiveservices account show' does not support --ids, so pass the name parsed off the resource id.
legacy_account_name = legacy_account_id.rstrip('/').split('/')[-1]
acct = utils.run(f"az cognitiveservices account show -g {resource_group_name} -n {legacy_account_name}",
                 "Resolved classic AOAI endpoint", "Failed to resolve classic AOAI endpoint")
legacy_endpoint = acct.json_data['properties']['endpoint'].rstrip('/')
utils.print_info(f"Classic AOAI endpoint: {legacy_endpoint}")
utils.print_info(f"disableLocalAuth: {acct.json_data['properties'].get('disableLocalAuth')}")

# Account key for scenario 2. With disableLocalAuth=true the listKeys call itself is blocked,
# which is half of what that scenario demonstrates - fall back to a placeholder so the call still runs.
keys = utils.run(f"az cognitiveservices account keys list -g {resource_group_name} -n {legacy_account_name}",
                 "Listed AOAI account keys", "Could not list AOAI account keys (expected when disableLocalAuth=true)")
legacy_api_key = keys.json_data.get('key1') or 'local-auth-disabled-placeholder-key'

# Entra data-plane token, used ONLY by the direct scenarios. APIM mints its own token from its managed identity.
tok = utils.run("az account get-access-token --resource https://cognitiveservices.azure.com",
                "Acquired Entra data-plane token", "Failed to acquire Entra data-plane token")
entra_token = tok.json_data.get('accessToken', '')

deployment = legacy_foundry_config["deploymentName"]
apim_base = f"{apim_resource_gateway_url}/{inference_api_path}"
chat_url_direct = f"{legacy_endpoint}/openai/deployments/{deployment}/chat/completions?api-version={inference_api_version}"
question = "which model are you using?"
chat_body = {"model": deployment, "messages": [{"role": "user", "content": question}]}
resp_body = {"model": deployment, "input": question}

utils.print_info(f"Deployment: {deployment}")
utils.print_info(f"APIM Base: {apim_base}")


def scenario(n, title, url, headers, body, auth_label, expected):
    print(f"\n{'─' * 108}")
    print(f"{n}) {title}")
    print(f"   POST      {url}")
    print(f"   auth      {auth_label}")
    print(f"   expected  {expected}")
    try:
        r = requests.post(url, headers={**headers, "Content-Type": "application/json"}, json=body, timeout=120)
    except Exception as e:
        utils.print_error(f"   transport error: {e}")
        return

    colour = utils.BOLD_GREEN if r.ok else utils.BOLD_RED
    print(f"   actual    {colour}HTTP {r.status_code} {r.reason}{utils.RESET_FORMATTING}"
          f"   x-ms-region: {r.headers.get('x-ms-region', '—')}")

    # WWW-Authenticate is required on a real 401 challenge and absent on a 403 policy refusal,
    # so it distinguishes "no identity" from "credential type forbidden".
    for h in ("x-ms-error-code", "WWW-Authenticate", "apim-request-id"):
        if r.headers.get(h):
            print(f"   {h:<9} {r.headers[h][:120]}")

    try:
        data = r.json()
    except ValueError:
        print(f"   body      {r.text[:200]}")
        return

    if r.ok:
        if "choices" in data:  # Chat Completions
            text = data["choices"][0]["message"]["content"]
        else:                  # Responses
            text = " ".join(c.get("text", "") for item in data.get("output", [])
                            for c in item.get("content", []) if c.get("type") == "output_text")
        print(f"   served    {data.get('model', '?')}")
        print(f"   💬        {text.strip()[:200]}")
    else:
        err = data.get("error", data) if isinstance(data, dict) else {}
        print(f"   error     {err.get('code', '')} — {str(err.get('message', r.text))[:200]}")


# 1-4 go straight to the classic account, bypassing APIM entirely.
scenario(1, "Direct to classic AOAI — Chat Completions, NO credential",
         chat_url_direct, {}, chat_body,
         "none", "401 — no identity established")

# A wrong key VALUE is documented to return 401; a 403 here means the gateway refused the
# credential TYPE without validating it, i.e. disableLocalAuth is being enforced.
scenario(2, "Direct to classic AOAI — Chat Completions, AOAI account key (local auth)",
         chat_url_direct, {"api-key": legacy_api_key}, chat_body,
         "api-key: <AOAI account key>", "403 (docs say 401) — key auth disabled by disableLocalAuth=true")

scenario(3, "Direct to classic AOAI — Chat Completions, Entra token",
         chat_url_direct, {"Authorization": f"Bearer {entra_token}"}, chat_body,
         "Authorization: Bearer <Entra token>", "200 with 'Cognitive Services OpenAI User', else 401/403")

scenario(4, "Direct to classic AOAI — Responses, Entra token",
         f"{legacy_endpoint}/openai/responses?api-version={inference_api_version}",
         {"Authorization": f"Bearer {entra_token}"}, resp_body,
         "Authorization: Bearer <Entra token>", "200 with 'Cognitive Services OpenAI User', else 401/403")

# 5 & 6 go through APIM, which routes on the body "model" field and swaps the credential.
scenario(5, "Via APIM — Chat Completions, APIM subscription key",
         f"{apim_base}/openai/deployments/{deployment}/chat/completions?api-version={inference_api_version}",
         {"api-key": api_key}, chat_body,
         "api-key: <APIM subscription key>", "200 — APIM re-auths to the backend with its managed identity")

scenario(6, "Via APIM — Responses, APIM subscription key",
         f"{apim_base}/openai/responses?api-version={inference_api_version}",
         {"api-key": api_key}, resp_body,
         "api-key: <APIM subscription key>", "200 — deployment selected by body 'model' only")

print(f"\n{'─' * 108}")
utils.print_info("Scenarios 2, 5 and 6 all send an 'api-key' header — but 2 carries an AOAI account key and 5/6 carry an APIM subscription key.")
utils.print_info("Deployment name is in the URL path for Chat Completions, and only in the body 'model' for Responses.")


### 🧩 Does the AOAI connector proxy inference? — empirical test

`foundry5` is an AIServices Foundry resource with **no model deployments of its own**. Its project holds an `AzureOpenAI` connection (`aoai-connector-foundry4`) pointing at the classic account that owns `legacy-gpt-4o` ([foundry5-connected-openai.bicep](foundry5-connected-openai.bicep#L70)).

The question this cell settles: **does adding that connection make `legacy-gpt-4o` callable on `foundry5`'s inference endpoints?**

**Answer, confirmed against a live deployment: no — on either endpoint.**

Each endpoint gets a matched pair: the connected deployment name, and a control name that exists nowhere.

| # | Endpoint | `model` | Token audience | Observed |
|---|---|---|---|---|
| 7 | **resource** | `legacy-gpt-4o` | `cognitiveservices.azure.com` | ✅ **404** |
| 8 | **resource** | `deployment-that-does-not-exist` *(control for 7)* | `cognitiveservices.azure.com` | ✅ **404** — identical to 7 |
| 9 | **project** | `legacy-gpt-4o` | `cognitiveservices.azure.com` | ✅ **401** — wrong audience for this endpoint |
| 10 | **project** | `legacy-gpt-4o` | `ai.azure.com` | ✅ **404** — auth passed, lookup still failed |
| 11 | **project** | `deployment-that-does-not-exist` *(control for 10)* | `ai.azure.com` | ✅ **404** — identical to 10 |
| — | control plane | — | — | `foundry5` deployments **empty**, connection object present |

**The controls are what make this proof rather than anecdote.** A lone 404 could be a typo, a propagation delay, or a malformed request. Because `legacy-gpt-4o` and a name that was never created anywhere produce the *same* status on the *same* endpoint, the connected deployment demonstrably occupies no privileged position in `foundry5`'s namespace. The connection imported nothing — on the resource endpoint **and** the project endpoint.

**Scenarios 9 and 10 are identical except for the token audience**, and together they demonstrate something the documentation does not state plainly:

> **The correct Entra audience depends on the endpoint you call, not on the resource that owns it.**
> `*.openai.azure.com` accepts `https://cognitiveservices.azure.com/.default`.
> `*.services.ai.azure.com` (the project endpoint) requires `https://ai.azure.com/.default` and returns **401** for the other.

That pairing is also what makes scenario 10 conclusive. Had 10 returned 401 or 403, you could not separate *"auth blocked me"* from *"the deployment isn't there."* Because 10 authenticated successfully and **still** returned 404, the failure is unambiguously deployment resolution.

**Scenario 10 answers an undocumented question.** No published Microsoft page confirms or denies that the project-scoped OpenAI route enumerates connected-account deployments. The observed 404 settles it: **it does not.** The connection is discovery and auth metadata on both endpoints.

> **These calls use v1 (`/openai/v1/responses`, no `api-version`)** rather than the dated preview form used elsewhere in this notebook, because the project endpoint is a v1-shaped surface.

> **If you re-run in a fresh subscription and scenarios 10–11 return 403**, the deployment question is unanswered for that run — auth precedes deployment resolution. [foundry5-connected-openai.bicep](foundry5-connected-openai.bicep#L84) grants the deployer *Azure AI Project Manager*, which is not necessarily a data-plane inference role. Grant `Cognitive Services OpenAI User` on `foundry5` and retry:
> ```
> az role assignment create --assignee <your-object-id> --role "Cognitive Services OpenAI User" --scope <foundry5AccountId>
> ```


In [ ]:
# Reuses scenario(), entra_token and question from the cell above.

foundry5_account_id = utils.get_deployment_output(dep, 'foundry5AccountId', 'Foundry5 account id')
foundry5_project_id = utils.get_deployment_output(dep, 'foundry5ProjectId', 'Foundry5 project id')
foundry5_project_endpoint = utils.get_deployment_output(dep, 'foundry5ProjectEndpoint', 'Foundry5 project endpoint')
foundry5_account_name = foundry5_account_id.rstrip('/').split('/')[-1]

f5 = utils.run(f"az cognitiveservices account show -g {resource_group_name} -n {foundry5_account_name}",
               "Resolved foundry5 account", "Failed to resolve foundry5 account")
f5_endpoints = f5.json_data.get('properties', {}).get('endpoints') or {}
foundry5_openai_endpoint = next(
    (v for k, v in f5_endpoints.items() if 'openai' in k.lower()),
    f"https://{foundry5_account_name}.openai.azure.com").rstrip('/')
utils.print_info(f"Foundry5 OpenAI endpoint: {foundry5_openai_endpoint}")

# The *.services.ai.azure.com project surface rejects the cognitiveservices audience, so mint a second token.
tok_ai = utils.run("az account get-access-token --resource https://ai.azure.com",
                   "Acquired Entra token for https://ai.azure.com", "Failed to acquire ai.azure.com token")
entra_token_ai = tok_ai.json_data.get('accessToken', '')

absent_model = "deployment-that-does-not-exist"
resource_responses_url = f"{foundry5_openai_endpoint}/openai/v1/responses"
project_responses_url = f"{foundry5_project_endpoint.rstrip('/')}/openai/v1/responses"

# 7 & 8 — RESOURCE endpoint: connected deployment vs. a name that exists nowhere.
# v1 surface: no api-version, deployment selected by body "model".
scenario(7, "foundry5 RESOURCE endpoint — Responses, model = legacy-gpt-4o (lives on foundry4)",
         resource_responses_url,
         {"Authorization": f"Bearer {entra_token}"},
         {"model": deployment, "input": question},
         "Bearer <cognitiveservices.azure.com>",
         "404 — connection does NOT import the deployment")

scenario(8, f"foundry5 RESOURCE endpoint — Responses, model = {absent_model} (CONTROL for 7)",
         resource_responses_url,
         {"Authorization": f"Bearer {entra_token}"},
         {"model": absent_model, "input": question},
         "Bearer <cognitiveservices.azure.com>",
         "404 — should be IDENTICAL to scenario 7")

# 9 vs 10: same URL and body, only the token AUDIENCE differs.
scenario(9, "foundry5 PROJECT endpoint — Responses, WRONG audience (cognitiveservices.azure.com)",
         project_responses_url,
         {"Authorization": f"Bearer {entra_token}"},
         {"model": deployment, "input": question},
         "Bearer <cognitiveservices.azure.com>",
         "401 — project surface rejects this audience")

scenario(10, "foundry5 PROJECT endpoint — Responses, CORRECT audience (ai.azure.com)",
         project_responses_url,
         {"Authorization": f"Bearer {entra_token_ai}"},
         {"model": deployment, "input": question},
         "Bearer <ai.azure.com>",
         "404 — project route does NOT resolve connected deployments either")

# 11 mirrors 8 on the project surface, so both surfaces get the same control.
scenario(11, f"foundry5 PROJECT endpoint — Responses, model = {absent_model} (CONTROL for 10)",
         project_responses_url,
         {"Authorization": f"Bearer {entra_token_ai}"},
         {"model": absent_model, "input": question},
         "Bearer <ai.azure.com>",
         "404 — should be IDENTICAL to scenario 10")

# Control plane: the connection exists, but foundry5 owns no deployments.
print(f"\n{'─' * 108}")
print("Control plane — what foundry5 actually owns vs. what it merely references")

deps5 = utils.run(f"az cognitiveservices account deployment list -g {resource_group_name} -n {foundry5_account_name}",
                  "Listed foundry5 deployments", "Failed to list foundry5 deployments")
names5 = [d.get('name') for d in (deps5.json_data if isinstance(deps5.json_data, list) else [])]
print(f"   foundry5 deployments : {names5 or '[] (none — foundry5 owns no models)'}")

conns = utils.run(
    f'az rest --method get --url "https://management.azure.com{foundry5_project_id}/connections?api-version=2025-04-01-preview"',
    "Listed foundry5 project connections", "Failed to list foundry5 project connections")
for c in (conns.json_data.get('value') or []):
    p = c.get('properties', {})
    print(f"   connection           : {c.get('name')}  category={p.get('category')}  authType={p.get('authType')}")
    print(f"   -> target            : {p.get('target')}")

print(f"\n{'─' * 108}")
utils.print_info("7==8 and 10==11: on BOTH surfaces the connected deployment is indistinguishable from one that never existed.")
utils.print_info("9 vs 10 differ only by token audience: *.openai.azure.com takes cognitiveservices, *.services.ai.azure.com takes ai.azure.com.")
utils.print_info("The connection is control-plane discovery: it references foundry4's deployment, it does not host or proxy it.")


<a id='metrics'></a>
### 📊 Visualize token & per-resource metrics

After running the test cells above, query the Log Analytics workspace to see:
- **Token usage by requested model** — emitted by the APIM `azure-openai-emit-token-metric` policy.
- **Per-Foundry-resource metrics** — `AllMetrics` diagnostics from each Cognitive Services (Foundry) resource.
- **Per-request LLM logs** — `ApiManagementGatewayLlmLog` rows with prompt/completion token counts and the actual `DeploymentName` served (handy for confirming the `gpt-5-mini` PTU→TPM spillover).

> Telemetry takes ~2–5 minutes to ingest. `log_analytics_id` comes from the deployment outputs (step 3). The first run may prompt the Azure CLI to install the `log-analytics` extension.


In [ ]:
# Query the Log Analytics workspace for telemetry produced by the test cells above.
# log_analytics_id (workspace GUID) comes from the deployment outputs (step 3).
# Allow ~2-5 minutes after running the tests for logs/metrics to ingest.

queries = {
    # Token usage emitted by the APIM azure-openai-emit-token-metric policy.
    # Workspace-based App Insights lands custom metrics in AppMetrics (classic App Insights: use customMetrics).
    "Token usage by requested model (APIM emit-token-metric)":
        "AppMetrics "
        "| where TimeGenerated > ago(1h) "
        "| where Name in ('Total Tokens','Prompt Tokens','Completion Tokens') "
        "| extend RequestedModel = tostring(Properties['Requested Model']) "
        "| summarize Tokens = sum(Sum) by RequestedModel, Name "
        "| order by RequestedModel asc, Name asc",
    # Per-Foundry-resource platform metrics (AllMetrics diagnostic setting on each Cognitive Services account).
    "Per-Foundry-resource metrics (Cognitive Services AllMetrics)":
        "AzureMetrics "
        "| where TimeGenerated > ago(1h) "
        "| where ResourceProvider == 'MICROSOFT.COGNITIVESERVICES' "
        "| summarize Total = sum(Total) by Resource, MetricName "
        "| order by Resource asc, MetricName asc",
    # Per-request LLM logs with token counts (APIM largeLanguageModel diagnostic -> ApiManagementGatewayLlmLog).
    # DeploymentName reveals the actual backend deployment served (e.g. gpt-5-mini-ptu vs gpt-5-mini-tpm on spillover).
    "Recent per-request LLM logs with token counts (ApiManagementGatewayLlmLog)":
        "ApiManagementGatewayLlmLog "
        "| where TimeGenerated > ago(1h) "
        "| extend TotalTokens = PromptTokens + CompletionTokens "
        "| project TimeGenerated, DeploymentName, ModelName, PromptTokens, CompletionTokens, TotalTokens, IsStreamCompletion "
        "| order by TimeGenerated desc "
        "| take 20",
}

for title, kql in queries.items():
    print(f"\n=== {title} ===")
    out = utils.run(f'az monitor log-analytics query -w {log_analytics_id} --analytics-query "{kql}" -o json',
                    "Query succeeded", "Query failed")
    rows = out.json_data if isinstance(out.json_data, list) else []
    if not rows:
        print("(no rows yet — give telemetry a few minutes after running the test cells)")
        continue
    cols = list(rows[0].keys())
    widths = {c: max(len(c), *(len(str(r.get(c, ''))) for r in rows)) for c in cols}
    print(" | ".join(c.ljust(widths[c]) for c in cols))
    print("-+-".join("-" * widths[c] for c in cols))
    for r in rows:
        print(" | ".join(str(r.get(c, '')).ljust(widths[c]) for c in cols))

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.